In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

In [2]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../raw_data/network_architecture_EDIN_DETAILED.txt")
docs = loader.load()

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/var/folders/5p/c3nk5nb53p5bkrln79yrc93r0000gn/T/ipykernel_87723/2347249996.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
/var/folders/5p/c3nk5nb53p5bkrln79yrc93r0000gn/T/ipykernel_87723/2347249996.py:6: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


In [7]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder: 100%|██████████| 1/1 [00:00<00:00, 267.46it/s]
Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.
Generating Samples: 100%|██████████| 10/10 [00:28<00:00,  2.85s/it]


In [9]:
dataset.to_pandas()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,"Within the context of the EDIN architecture, w...",[=============================================...,"In the EDIN architecture, the Metro Bridge (MB...",Network Operations Engineer,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
1,What is the network topology and node assignme...,[2. NETWORK TOPOLOGY DESCRIPTION\n============...,"In the described network topology, EDINBURGH i...",Network Operations Engineer,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer
2,wut is dslam and how it connecs to cpe?,[3. NETWORK ELEMENT: CPE (Customer Premise Equ...,The DSLAM is the device to which the CPE WAN i...,Network Operations Engineer,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
3,What DSLAM do?,[4. NETWORK ELEMENT: DSLAM\n==================...,"DSLAM is the main access aggregation node, ser...",Network Operations Engineer,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
4,What is the primary function of an MB node in ...,[5. NETWORK ELEMENT: METRO BRIDGE (MB NODE)\n=...,The MB node aggregates all DSLAM uplinks withi...,Network Operations Engineer,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
5,How is Grafana used in post-change verificatio...,[<1-hop>\n\nor access to premises - Approved m...,Grafana is used in post-change verification to...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
6,What are the key hardware features and redunda...,[<1-hop>\n\n5. NETWORK ELEMENT: METRO BRIDGE (...,The EDIN-MB-6500 Metro Bridge node features 48...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
7,When troubleshooting a customer line down inci...,[<1-hop>\n\n2. Obstructed airflow (cable manag...,During troubleshooting of a customer line down...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
8,Wut MB node do in DSLAM netwrok and how does i...,[<1-hop>\n\n4. NETWORK ELEMENT: DSLAM\n=======...,The MB (Metro Bridge) node in the DSLAM networ...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,If MC hardware fail cause BGP/MPLS instability...,[<1-hop>\n\nresolved ─────────────────────────...,If MC hardware fail and cause BGP/MPLS instabi...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer


In [8]:
from app.services.tools import _rag_graph

LangSmith tracing enabled. Project: Agentic_Network_Assistance


In [10]:
for test_row in dataset:
  response = _rag_graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

In [11]:
dataset.samples[0].eval_sample.response

'The role of the Metro Bridge (MB) node in the end-to-end connection model within the EDIN architecture is to aggregate all DSLAM uplinks within a metropolitan area and facilitate the transition from access Ethernet to the Metro Core MPLS network. The MB node connects upstream to the Metro Core (MC) nodes and supports up to 10 DSLAM connections, as well as 2 uplinks to the MC node. \n\nIn the overall network structure, the MB node fits between the Customer Premise Equipment (CPE) and the Metro Core (MC) nodes. Specifically, the DSLAMs connect to the MB node, which then connects to the MC nodes, enabling the flow of data from the CPE through the DSLAM aggregation, into the MB, and onward to the MC for further routing towards the Peta Core and Datacenter (DC) termination.'

In [13]:
from ragas import EvaluationDataset

df = dataset.to_pandas()
df = df.dropna(axis=1, how="all")   # drop fully-empty columns
df = df.fillna("")                   # fill remaining NaN with empty string
evaluation_dataset = EvaluationDataset.from_list(df.to_dict(orient="records"))

In [14]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

/var/folders/5p/c3nk5nb53p5bkrln79yrc93r0000gn/T/ipykernel_87723/3221782877.py:4: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))


In [15]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

baseline_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
baseline_result

/var/folders/5p/c3nk5nb53p5bkrln79yrc93r0000gn/T/ipykernel_87723/646324422.py:1: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
/var/folders/5p/c3nk5nb53p5bkrln79yrc93r0000gn/T/ipykernel_87723/646324422.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
/var/folders/5p/c3nk5nb53p5bkrln79yrc93r0000gn/T/ipykernel_87723/646324422.py:1: DeprecationWarning: Importing FactualCorrectn

{'context_recall': 0.6888, 'faithfulness': 0.9009, 'factual_correctness(mode=f1)': 0.5240, 'answer_relevancy': 0.8388, 'context_entity_recall': 0.2361, 'noise_sensitivity(mode=relevant)': 0.3960}

In [23]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env", override=True)  # reload .env with new key

import importlib
import app.services.tools as tools_module
importlib.reload(tools_module)
from app.services.tools import _advanced_rag_graph

LangSmith tracing enabled. Project: Agentic_Network_Assistance


In [24]:
from app.services.tools import _advanced_rag_graph

In [25]:
import time
import copy

rerank_dataset = copy.deepcopy(dataset)

for test_row in rerank_dataset:
  response = _advanced_rag_graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
  time.sleep(6)

In [26]:
rerank_dataset.samples[0].eval_sample.response

'In the context of the EDIN architecture, the role of the Metro Bridge (MB) node in the end-to-end connection model is to aggregate all DSLAM uplinks within a metropolitan area and provide the transition from access Ethernet to the Metro Core MPLS network. The MB node fits between the DSLAM aggregation and the Metro Core (MC) nodes, serving as a critical point for DSLAM aggregation and VLAN tagging. It connects to multiple DSLAMs (up to 10) and has uplinks to the MC node (up to 2), facilitating the flow of data from the customer premise equipment (CPE) through the DSLAMs and into the Metro Core for further routing towards the Datacenter (DC).'

In [29]:
rerank_df = dataset.to_pandas()
rerank_df = rerank_df.dropna(axis=1, how="all")
rerank_df = rerank_df.fillna("")
rerank_evaluation_dataset = EvaluationDataset.from_list(rerank_df.to_dict(orient="records"))

In [30]:
rerank_result = evaluate(
    dataset=rerank_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
rerank_result

Evaluating:   2%|▏         | 1/60 [00:02<02:05,  2.12s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 60/60 [04:44<00:00,  4.75s/it]


{'context_recall': 0.6888, 'faithfulness': 0.8697, 'factual_correctness(mode=f1)': 0.5470, 'answer_relevancy': 0.8411, 'context_entity_recall': 0.2933, 'noise_sensitivity(mode=relevant)': 0.3703}